# CS604 Colab launcher — Ana: which parameters to train? (pilot scale)

Same budget (~4.0M trainable, ~4.6%), different rules for choosing *which* parameters. Arms in `configs/param-selection_336px_same-budget_1k-class-pilot.yaml`, on the same 1,000-class pilot subset and holdout as the earlier ablations:

| Arm | Rule | Source |
|---|---|---|
| A | LoRA r24 everywhere (uniform low rank) | current recipe, 4.65% |
| F_llrd / F_flat | full FT (ceiling); Van's LLRD recipe / one lr 3e-5 | 100% |
| S0 / S5 / S11 | block *i*'s attention at full rank + LoRA r8 | Lee et al., surgical FT |
| Y_layer / Y_matrix | largest fine-tuned angle in F_flat: one ATTN layer (+ LoRA r8) / single matrices | Ye et al., partial FT |
| G_k42 / G_k10 | top-K input connections per neuron by gradient | Zhang et al., GPS |

Three phases: (1) A, F, S; (2) select the Y layers from F_flat's checkpoint and the G masks from gradients; (3) Y and G. Finished arms are skipped if you Run all again.

About 1 h 15 min on an A100: data ~15 min, prototypes ~5 min, 11 arms × ~5–6 min, selection ~5 min. Results go to `CS604_RUNS/ablation_ablation_param_selection/`.

Runtime: **A100 GPU**. Use **Runtime → Run all**.


In [ ]:
REPO_URL = "https://github.com/keaidewoq/cs604-project.git"
REPO_DIR = "/content/repo"
DATA_DIR = "/content/data"
# Must match a folder (or shortcut) in My Drive — not the browser URL.
DRIVE_OUT = "/content/drive/MyDrive/CS604_RUNS"
BRANCH = "ana-branch"
METHOD = "lora_r24_336px_qkv-proj-fc_extras_proto-cosine_2.4h"  # base config; only used to build prototypes_336.pt if it is missing
PROTO_PT = f"{DRIVE_OUT}/prototypes_336.pt"
SPEC = "configs/param-selection_336px_same-budget_1k-class-pilot.yaml"
PILOT_DATA_DIR = "/content/pilot_data"
FEATURE_DIR = "/content/features_pilot"  # local disk
# End the runtime after the last cell, or as soon as training fails.
AUTO_DISCONNECT = True


def end_runtime(reason):
    """Flush Drive writes, then release the GPU so it stops using compute units."""
    print(reason, flush=True)
    if not AUTO_DISCONNECT:
        return
    from google.colab import drive, runtime
    try:
        drive.flush_and_unmount()  # without this the last checkpoint can be lost
        print("Drive flushed and unmounted", flush=True)
    except Exception as error:
        print(f"Drive flush failed: {error}", flush=True)
    runtime.unassign()

## 1. Clone or pull the repo

In [ ]:
import os
from getpass import getpass

token = getpass("GitHub PAT (hidden; repo scope). Leave empty if the repo is public: ")
clone_url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    if token:
        !git -C {REPO_DIR} pull {clone_url} {BRANCH}
    else:
        !git -C {REPO_DIR} pull origin {BRANCH}
else:
    !git clone -b {BRANCH} {clone_url} {REPO_DIR}

%cd {REPO_DIR}
!git branch --show-current
!git rev-parse --short HEAD

## 2. Install requirements

In [ ]:
%pip install -r requirements.txt

## 3. Mount Drive and check `CS604_RUNS`

If this cell errors, add a shortcut named `CS604_RUNS` to My Drive from the [shared folder](https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR).

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

out = Path(DRIVE_OUT)
if not out.is_dir():
    raise FileNotFoundError(
        f"{DRIVE_OUT} not found after mounting.\n"
        "Open https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR "
        "→ Organize → Add shortcut to Drive → My Drive, and name the shortcut CS604_RUNS."
    )
out.mkdir(parents=True, exist_ok=True)
print("Writing runs to", out.resolve())
print("Existing items:", [p.name for p in out.iterdir()][:20])

## 4. Download data to `/content/data`

In [ ]:
!python scripts/download_data.py --dest {DATA_DIR}

## 4b. Build class prototypes at 336 (once)

Same as Van's prototype head, but built from 336 features (built with `configs/{METHOD}.yaml`, so at its `img_size`). Run **once**; about 20–25 min on an A100. Takes a while (embeds train images). Skip if the file already exists on Drive.

In [ ]:
from pathlib import Path

if Path(PROTO_PT).is_file():
    print("Already have", PROTO_PT)
else:
    !python scripts/build_prototypes.py --config configs/{METHOD}.yaml --data_root {DATA_DIR} --output {PROTO_PT}

## 5. Phase 1: A, full FT, surgical arms


In [ ]:
import subprocess
import sys
from pathlib import Path

OUT_DIR = Path(DRIVE_OUT) / f"ablation_{Path(SPEC).stem}"
PILOT_ROOT = f"{PILOT_DATA_DIR}/c1000_h5_s0"  # built by scripts/ablation.py from the spec's pilot block


def run(cmd, what):
    proc = subprocess.run([sys.executable, "-u", *cmd])
    if proc.returncode != 0:
        end_runtime(f"{what} failed (exit {proc.returncode}); see the output above")
        raise SystemExit(f"{what} failed")


def ablation(*only):
    cmd = ["scripts/ablation.py", "--spec", SPEC, "--data_root", DATA_DIR, "--output_dir", DRIVE_OUT,
           "--pilot_data_dir", PILOT_DATA_DIR, "--feature_dir", FEATURE_DIR]
    run(cmd + (["--only", *only] if only else []), "ablation")

ablation("A_full_recipe", "F_full_ft_llrd", "F_full_ft_flat",
         "S0_attn_block0", "S5_attn_block5", "S11_attn_block11")


## 6. Phase 2: choose the Y layers (fine-tuned angle) and the G masks (gradients)

Angles come from `F_full_ft_flat` (one lr for all layers, so early layers are not shrunk by layer decay). Gradients are summed over 100 batches of the pilot's fit images, never the holdout.


In [ ]:
import json

ft_run = json.loads((OUT_DIR / "arms" / "F_full_ft_flat.json").read_text())["run_dir"]
if not (OUT_DIR / "selected_matrix.yaml").is_file():
    run(["scripts/select_params.py", "angle", "--config", "configs/lora_r24_336px_qkv-proj-fc_extras_proto-cosine_2.4h.yaml",
         "--ckpt", f"{ft_run}/last.pt", "--out_dir", str(OUT_DIR)], "angle selection")
if not (OUT_DIR / "gps_mask_k10.pt").is_file():
    run(["scripts/select_params.py", "gradient", "--config", "configs/lora_r24_336px_qkv-proj-fc_extras_proto-cosine_2.4h.yaml",
         "--data_root", PILOT_ROOT, "--prototype_file", str(OUT_DIR / "prototypes_pilot_fit.pt"),
         "--k", "42", "10", "--out_dir", str(OUT_DIR)], "gradient selection")
print("selected_layer: ", (OUT_DIR / "selected_layer.yaml").read_text())
print("selected_matrix:", (OUT_DIR / "selected_matrix.yaml").read_text())


## 7. Phase 3: Y and G arms (A, F, S are reused)


In [ ]:
ablation()


## 8. Results

`delta_vs_A_points`: pilot holdout top-1 against A, in percentage points. Treat under ~0.3 as a tie (one seed).

- **S0 vs S5 vs S11**: where full rank helps most (the depth screen predicts S0).
- **Y vs S**: is a full FT run a better guide than depth?
- **G vs Y_matrix**: single weights vs whole matrices at the same budget.
- **F_llrd**: the ceiling every arm is judged against.


In [ ]:
import pandas as pd

table = pd.read_csv(OUT_DIR / "ablation.csv")
display(table[["arm", "final_val_acc", "delta_vs_A_points", "best_val_acc", "early_dip",
               "backbone_pct", "training_min", "peak_vram_gb", "wall_min"]])

# fine-tuned angle by depth (degrees), from F_full_ft_flat: for the report
angles = pd.read_csv(OUT_DIR / "angles.csv")
angles["matrix"] = angles["module"].str.split(".").str[-1]
display(angles.pivot_table(index="block", columns="matrix", values="angle_deg", sort=False).round(3))


## 9. End the runtime


In [ ]:
end_runtime("all cells finished; ending the runtime")